In [ ]:
import torch

# 1. 数据准备 (Synthetic Data)
def create_data(num_samples=100):
    X = torch.randn(num_samples, 3)
    # 设定一个非线性的真实规律：y = x1^2 + x2 - 0.5x3 + 0.5
    y = X[:, 0:1]**2 + X[:, 1:2] - 0.5 * X[:, 2:3] + 0.5
    return X, y

# 2. 手写 MLP 类 (不使用 nn.Linear)
class ManualMLP:
    def __init__(self, in_features, hidden_features, out_features):
        # 初始化第一层权重 (Kaiming 初始化思想：除以特征数的平方根，防止梯度爆炸)
        self.w1 = torch.randn(in_features, hidden_features, requires_grad=True) * (2/in_features)**0.5
        self.b1 = torch.zeros(hidden_features, requires_grad=True)
        
        # 初始化第二层权重
        self.w2 = torch.randn(hidden_features, out_features, requires_grad=True) * (2/hidden_features)**0.5
        self.b2 = torch.zeros(out_features, requires_grad=True)
        
        self.params = [self.w1, self.b1, self.w2, self.b2]

    def forward(self, x):
        # 第一层：矩阵乘法 + 广播偏置 + ReLU
        # (Batch, In) @ (In, Hidden) -> (Batch, Hidden)
        z1 = torch.matmul(x, self.w1) + self.b1
        h1 = torch.relu(z1)
        
        # 第二层：输出层 (线性)
        # (Batch, Hidden) @ (Hidden, Out) -> (Batch, Out)
        out = torch.matmul(h1, self.w2) + self.b2
        return out

# 3. 训练配置
X, y_true = create_data(200)
model = ManualMLP(in_features=3, hidden_features=10, out_features=1)
learning_rate = 0.01
epochs = 500

# 4. 训练循环
for epoch in range(epochs):
    # --- 前向传播 ---
    y_pred = model.forward(X)
    
    # --- 计算 MSE Loss ---
    loss = torch.mean((y_pred - y_true)**2)
    
    # --- 梯度清零 ---
    for p in model.params:
        if p.grad is not None:
            p.grad.zero_()
            
    # --- 反向传播 ---
    loss.backward()
    
    # --- 参数更新 ---
    with torch.no_grad():
        for p in model.params:
            p -= learning_rate * p.grad
            
    # --- 日志打印 ---
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1:>3}/{epochs}] | Loss: {loss.item():.6f}")

print("\n训练完成！权重已优化。")